# 03. Random local editing

**Baseline**: scaffold-preserving random local edits with no descriptor guidance.

Design:
- Murcko scaffold atoms are **never** edited (replace / delete / bond change)
- No descriptor information is used during candidate generation
- The same candidate pool is evaluated against all 7 tasks
- `candidate_rank` = order of accepted candidates (invalid/scaffold_failed/duplicate excluded)
- Generation statistics are saved per parent for Validity / Scaffold retention / Uniqueness

Generation statistics use exclusive categories:
```
n_examined = n_invalid + n_scaffold_failed + n_duplicate + n_valid_unique
```
Processing order: attempted → valid? → scaffold_retained? → not_duplicate? → accepted


In [ ]:
import random, sys
import pandas as pd
from tqdm import tqdm
from rdkit import Chem, RDLogger
RDLogger.DisableLog("rdApp.*")
sys.path.append("..")

from configs.benchmark_config import (
    ACTIVE_PARENT_DATASET, RESULTS_DIR,
    N_CANDIDATES_PER_PARENT, RANDOM_SEED, TASKS,
)
from utils.parents import prepare_parent
from utils.scaffold import get_scaffold_atom_ids, scaffold_is_retained
from utils.records import make_candidate_record

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
random.seed(RANDOM_SEED)


In [ ]:
ATOM_TYPES = [6, 7, 8, 9, 16, 17]   # C N O F S Cl
N_ATTEMPTS  = N_CANDIDATES_PER_PARENT * 10
METHOD = "random"


In [ ]:
def random_edit_outside_scaffold(mol, scaffold_ids):
    """One random edit on a non-scaffold atom. Returns Mol or None."""    rw = Chem.RWMol(mol)
    free = [a.GetIdx() for a in rw.GetAtoms() if a.GetIdx() not in scaffold_ids]
    if not free:
        return None
    edit = random.choice(["replace", "add", "delete"])
    try:
        if edit == "replace":
            idx = random.choice(free)
            old = rw.GetAtomWithIdx(idx).GetAtomicNum()
            rw.GetAtomWithIdx(idx).SetAtomicNum(
                random.choice([x for x in ATOM_TYPES if x != old])
            )
        elif edit == "add":
            idx = random.choice(free)
            ni = rw.AddAtom(Chem.Atom(random.choice(ATOM_TYPES)))
            rw.AddBond(idx, ni, Chem.BondType.SINGLE)
        else:
            tf = [i for i in free if rw.GetAtomWithIdx(i).GetDegree() == 1]
            if not tf:
                return None
            rw.RemoveAtom(random.choice(tf))
        Chem.SanitizeMol(rw)
        return rw.GetMol()
    except Exception:
        return None


## Step 1: Smoke test on 5 parents

In [ ]:
df = pd.read_csv(ACTIVE_PARENT_DATASET)
if "canonical_smiles" not in df.columns and "smiles" in df.columns:
    df = df.rename(columns={"smiles": "canonical_smiles"})
if "parent_id" not in df.columns and "chembl_id" in df.columns:
    df = df.rename(columns={"chembl_id": "parent_id"})
print(f"Parents: {len(df)}")

records_test, stats_test = [], []
for _, row in df.head(5).iterrows():
    prepared = prepare_parent(row)
    if prepared is None:
        continue
    parent_id, parent_smiles, parent_desc, _ = prepared
    parent_mol = Chem.MolFromSmiles(parent_smiles)
    scaffold_ids = get_scaffold_atom_ids(parent_mol)

    # Explicit counters — no residual calculation
    n_examined = 0; n_invalid = 0; n_scaffold_failed = 0; n_duplicate = 0
    candidates = []; seen = set()

    for _ in range(N_ATTEMPTS):
        if len(candidates) >= N_CANDIDATES_PER_PARENT:
            break
        n_examined += 1
        cand_mol = random_edit_outside_scaffold(parent_mol, scaffold_ids)
        if cand_mol is None:
            n_invalid += 1          # invalid: RDKit parse/sanitize failed
            continue
        cand_smi = Chem.MolToSmiles(cand_mol)
        if not scaffold_is_retained(parent_mol, cand_mol):
            n_scaffold_failed += 1  # valid but scaffold lost
            continue
        if cand_smi == parent_smiles or cand_smi in seen:
            n_duplicate += 1        # valid, scaffold-retained, but duplicate/parent
            continue
        seen.add(cand_smi)
        candidates.append(cand_smi)

    n_valid_unique = len(candidates)
    assert n_invalid + n_scaffold_failed + n_duplicate + n_valid_unique == n_examined, (
        f"Stats sum mismatch: {n_invalid}+{n_scaffold_failed}+{n_duplicate}+{n_valid_unique} != {n_examined}"
    )

    stats_test.append({
        "method": METHOD, "parent_id": parent_id,
        "n_examined":      n_examined,
        "n_invalid":        n_invalid,
        "n_scaffold_failed": n_scaffold_failed,
        "n_duplicate":      n_duplicate,
        "n_valid_unique":   n_valid_unique,
    })
    for task_name in TASKS:
        for rank, cand_smi in enumerate(candidates, start=1):
            rec = make_candidate_record(
                method=METHOD, parent_id=parent_id,
                parent_smiles=parent_smiles, task_name=task_name,
                candidate_rank=rank, candidate_smiles=cand_smi,
                parent_desc=parent_desc,
            )
            if rec is not None:
                records_test.append(rec)

df_test = pd.DataFrame(records_test)
df_stats_test = pd.DataFrame(stats_test)
print(f"Records: {len(df_test)}  Tasks: {df_test['task'].nunique()}/7")
print(f"scaffold_retained all True: {df_test['scaffold_retained'].all()}")
print(f"candidate_rank range: {df_test['candidate_rank'].min()}–{df_test['candidate_rank'].max()}")
print()
print("Generation stats:")
print(df_stats_test.to_string())
print()
# Verify metrics
row0 = df_stats_test.iloc[0]
validity   = (row0.n_examined - row0.n_invalid) / row0.n_examined
sc_ret     = (row0.n_duplicate + row0.n_valid_unique) / (row0.n_examined - row0.n_invalid)
uniqueness = row0.n_valid_unique / (row0.n_duplicate + row0.n_valid_unique) if (row0.n_duplicate + row0.n_valid_unique) > 0 else float('nan')
print(f"Example ({row0.parent_id}): Validity={validity:.3f}  ScaffoldRetention={sc_ret:.3f}  Uniqueness={uniqueness:.3f}")


## Step 2: Full run — 1000 parents

In [ ]:
records, gen_stats = [], []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Random"):
    prepared = prepare_parent(row)
    if prepared is None:
        continue
    parent_id, parent_smiles, parent_desc, _ = prepared
    parent_mol = Chem.MolFromSmiles(parent_smiles)
    scaffold_ids = get_scaffold_atom_ids(parent_mol)

    n_examined = 0; n_invalid = 0; n_scaffold_failed = 0; n_duplicate = 0
    candidates = []; seen = set()

    for _ in range(N_ATTEMPTS):
        if len(candidates) >= N_CANDIDATES_PER_PARENT:
            break
        n_examined += 1
        cand_mol = random_edit_outside_scaffold(parent_mol, scaffold_ids)
        if cand_mol is None:
            n_invalid += 1
            continue
        cand_smi = Chem.MolToSmiles(cand_mol)
        if not scaffold_is_retained(parent_mol, cand_mol):
            n_scaffold_failed += 1
            continue
        if cand_smi == parent_smiles or cand_smi in seen:
            n_duplicate += 1
            continue
        seen.add(cand_smi)
        candidates.append(cand_smi)

    n_valid_unique = len(candidates)
    assert n_invalid + n_scaffold_failed + n_duplicate + n_valid_unique == n_examined

    gen_stats.append({
        "method": METHOD, "parent_id": parent_id,
        "n_examined":      n_examined,
        "n_invalid":        n_invalid,
        "n_scaffold_failed": n_scaffold_failed,
        "n_duplicate":      n_duplicate,
        "n_valid_unique":   n_valid_unique,
    })

    for task_name in TASKS:
        for rank, cand_smi in enumerate(candidates, start=1):
            rec = make_candidate_record(
                method=METHOD, parent_id=parent_id,
                parent_smiles=parent_smiles, task_name=task_name,
                candidate_rank=rank, candidate_smiles=cand_smi,
                parent_desc=parent_desc,
            )
            if rec is not None:
                records.append(rec)

print(f"Total records: {len(records)}")


In [ ]:
OUTPUT_FILE = f"{RESULTS_DIR}/random_local_editing.csv"
STATS_FILE  = f"{RESULTS_DIR}/random_generation_stats.csv"
pd.DataFrame(records).to_csv(OUTPUT_FILE, index=False)
pd.DataFrame(gen_stats).to_csv(STATS_FILE, index=False)
print(f"Saved → {OUTPUT_FILE}")
print(f"Saved → {STATS_FILE}")
